<a href="https://colab.research.google.com/github/adalves-ufabc/2026.Q3-PLN/blob/main/2026_Q3_PLN_AULA_12_Notebook_23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Processamento de Linguagem Natural [2026-Q3]**
Prof. Alexandre Donizeti Alves

## **Integração da API da Groq com o LangChain**
---



In [ ]:
#@title Instalando a biblioteca de integração com o LangChain

!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-groq]


In [ ]:
#@title Definindo a chave da API

from getpass import getpass

GROQ_API_KEY = getpass()

··········


**Integração**

In [ ]:
from langchain_groq import ChatGroq

modelo = ChatGroq(model="openai/gpt-oss-120b", api_key = GROQ_API_KEY)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

mensagens = [
    SystemMessage(content="Traduza o seguinte do Inglês para o Português"),
    HumanMessage(content="hi!"),
]

resposta = modelo.invoke(mensagens)

In [ ]:
resposta

AIMessage(content='Oi!', additional_kwargs={'reasoning_content': 'The user says "hi!" after system instructions. The developer says "Traduza o seguinte do Inglês para o Português". That seems like a meta instruction: translate the following from English to Portuguese. But the user just said "hi!". So we need to translate "hi!" to Portuguese. That would be "Oi!" or "Olá!". Should we respond with translation? Probably respond with "Oi!" or "Olá!". The user says hi! So we can respond with "Oi!" (Portuguese). Probably also ask if they need translation. But the developer instruction says "Traduza o seguinte do Inglês para o Português". The user gave English phrase "hi!". So we translate that. Output: "Oi!" (or "Olá!"). Let\'s do "Oi!".'}, response_metadata={'token_usage': {'completion_tokens': 170, 'prompt_tokens': 87, 'total_tokens': 257, 'completion_time': 0.358280732, 'completion_tokens_details': {'reasoning_tokens': 159}, 'prompt_time': 0.003795975, 'prompt_tokens_details': None, 'queue

In [ ]:
resposta.content

'Oi!'

**ChatPromptTemplate**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "Traduza o seguinte para {idioma}:"

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{texto}")]
)

prompt = prompt_template.invoke({"idioma": "italiano", "texto": "hi"})

prompt

ChatPromptValue(messages=[SystemMessage(content='Traduza o seguinte para italiano:', additional_kwargs={}, response_metadata={}), HumanMessage(content='hi', additional_kwargs={}, response_metadata={})])

In [ ]:
prompt.to_messages()

[SystemMessage(content='Traduza o seguinte para italiano:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hi', additional_kwargs={}, response_metadata={})]

In [ ]:
# envia o prompt ao modelo
resposta = modelo.invoke(prompt)

print(resposta.content)

Ciao.


In [ ]:
chain =  prompt_template | modelo

resposta = chain.invoke({"idioma": "português", "texto": "hi"})

In [ ]:
resposta

AIMessage(content='Oi (ou Olá)', additional_kwargs={'reasoning_content': 'User says "hi". The developer says: "Traduza o seguinte para português:" but no text is given. The user says "hi". Probably they want translation of "hi" to Portuguese. So answer: "Oi". Possibly also "Olá". Usually "hi" => "Oi". So respond with translation.'}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 83, 'total_tokens': 164, 'completion_time': 0.170068665, 'completion_tokens_details': {'reasoning_tokens': 66}, 'prompt_time': 0.030575381, 'prompt_tokens_details': None, 'queue_time': 0.222457597, 'total_time': 0.200644046}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_86dc843b12', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1799e-d49f-70d7-b0a2-c6ba067ede46-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 81, 'total_tokens': 164, 'output_token_deta

In [ ]:
resposta.content

'Oi (ou Olá)'

**Encadeando componentes com LCEL**

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [ ]:
chain = prompt_template | modelo | parser

In [ ]:
chain.invoke({"idioma": "portugues", "texto": "hi"})

'Oi'

**Cadeias**

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

In [ ]:
modelo = ChatGroq(model="openai/gpt-oss-120b", api_key = GROQ_API_KEY)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Me conte uma piada sobre {assunto}")
chain = prompt | modelo

In [ ]:
resposta = chain.invoke({"assunto": "papagaio"})

In [ ]:
from IPython.display import Markdown

Markdown(resposta.content)

Claro! Aqui vai:

— O papagaio entra num bar e pede uma cerveja.  
— O barman, surpreso, pergunta: “Você fala?”  
— O papagaio responde: “Claro que falo! Só não falo quando a conta vem!” 😄

Mais um exemplo:

In [ ]:
from langchain_groq import ChatGroq

modelo = ChatGroq(model="openai/gpt-oss-120b", api_key = GROQ_API_KEY)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Você é um assistente que traduz {idioma_input} para {idioma_output}.",
        ),
        ("human", "{input}"),
    ]
)

chain = prompt | modelo

resposta = chain.invoke(
    {
        "idioma_input": "Inglês",
        "idioma_output": "Português",
        "input": "I love programming.",
    }
)

In [ ]:
resposta.content

'Eu amo programar.'